# Nour — admin: index, evaluate, review feedback

Run this whenever the books (`backend/data/*.txt`) or the chunking settings change. No GPU needed (CPU runtime is fine): embeddings are computed by Cloudflare Workers AI (BGE-M3), the same model the site uses for questions.

**Before the first run**, add these in Colab **Secrets** (🔑 icon on the left, turn on *Notebook access* for each):

| Name | Where to find it |
|---|---|
| `SUPABASE_URL` | Supabase → Project Settings → API → Project URL |
| `SUPABASE_SERVICE_KEY` | Supabase → Project Settings → API Keys → **secret** key (or legacy `service_role`) |
| `CLOUDFLARE_ACCOUNT_ID` | Cloudflare dashboard → Workers & Pages → Account ID (right column) |
| `CLOUDFLARE_API_TOKEN` | Cloudflare → My Profile → API Tokens → Create → template **Workers AI** |
| `GROQ_API_KEY` | console.groq.com → API Keys (only needed for step 5) |

And run `backend/supabase/schema.sql`, then `backend/supabase/learning.sql`, once in the Supabase SQL Editor.

In [ ]:
# 1. Get the code (fresh copy each run)
#    BRANCH: "main" = what the live site runs; another branch = what its Vercel preview runs.
BRANCH = "main"
!rm -rf nour_project && git clone -q --depth 1 -b {BRANCH} https://github.com/bachar88/nour_project.git
%cd nour_project/backend
!git log --oneline -1
!pip install -q httpx python-dotenv

In [ ]:
# 2. Load the keys from Colab Secrets into environment variables (read-only: nothing is printed)
import os
from google.colab import userdata

for name in ["SUPABASE_URL", "SUPABASE_SERVICE_KEY", "CLOUDFLARE_ACCOUNT_ID", "CLOUDFLARE_API_TOKEN", "GROQ_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name)
        print("✓", name)
    except Exception:
        print("✗", name, "— missing (GROQ_API_KEY is only needed for step 5)")

In [ ]:
# 3. Index: chunk the books → BGE-M3 embeddings (Cloudflare) → Supabase
#    Skips if nothing changed. Add --force to rebuild, or --book akhdari for one book.
!python -m scripts.index_supabase

In [ ]:
# 4. Evaluate retrieval on the Salat test set (no Groq calls)
#    Look at the suggested MIN_COSINE_SCORE at the end.
!python -m scripts.evaluate

In [ ]:
# 5. (optional) The exam with real answers from Groq — about 2,000 tokens per question, ~4 minutes.
#    --save keeps the scores in Supabase (table eval_runs) so you can compare versions.
#    Change the label to describe what you changed since the last run.
!python -m scripts.evaluate --llm --save --label "describe this version"

## Test questions — what the RAG retrieves for each one

Run steps 1 and 2 first. Then:
- **T1** holds the questions (edit freely).
- **T2** shows, for each question, the 5 closest chunks with their score, which ones are **→ sent** to the model, the full text of the passages sent, and whether `MIN_COSINE_SCORE` would block the question. **No Groq calls, read-only.**
- **T3** (optional) runs the full pipeline exactly as the site does and prints the answer and its quotes. About 2,000 Groq tokens and 15 s per question.

How to read T2: if the right sentence is **not** among the chunks, it's a retrieval problem (chunking, thresholds). If it **is** in a passage marked *sent* but the answer is wrong, it's the model or prompt.

In [ ]:
# T1. Test questions
QUESTIONS = [
    # أوقات الصلاة
    "ما هو وقت صلاة الصبح؟",
    "ما هو الحد الفاصل بين وقت صلاة الظهر ووقت صلاة العصر؟",

    # أحكام الصلاة
    "هل يجوز للمرأة أن تؤم النساء أو الرجال؟",
    "ما هو حكم قصر الصلاة للمسافر؟",
    "ما العمل لمن نسي سجود السهو في الصلاة؟",

    # شروط الصلاة وفرائضها وسننها
    "ما هي فرائض الصلاة في المذهب المالكي؟",
    "ما هي سنن الصلاة الأكدة التي يسجد لسهوها؟",
    "ما حكم استقبال القبلة وستر العورة في الصلاة؟",

    # صلاة الجماعة والإمامة
    "هل يجوز للمرأة أن تؤم النساء أو الرجال في الصلاة؟",
    "ما حكم صلاة الجماعة في الفرض؟",
    "ماذا يفعل المسبوق إذا دخل مع الإمام في صلاة الجماعة؟",

    # قصر الصلاة والجمع والمسافر
    "ما هي المسافة المبيحة لقصر الصلاة للمسافر؟",
    "متى يبدأ المسافر في قصر الصلاة ومتى يتمها؟",
    "ما هي الحالات التي يجوز فيها جمع الصلاة؟",

    # سجود السهو
    "ما الفرق بين السجود القبلي والسجود البعدي في المذهب المالكي؟",
    "ما العمل لمن نسي سجود السهو القبلي حتى طال الفصل؟",
    "ما حكم من شك في عدد الركعات أثناء الصلاة؟",
]
print(len(QUESTIONS), "questions")

In [ ]:
# T2. Retrieved chunks for each question (read-only, no Groq calls)
import os, sys
sys.path.insert(0, os.getcwd())   # the backend/ folder (step 1 moved there)
from scripts.inspect_questions import make_engine, show_retrieval, show_answers

engine = make_engine(with_llm=False)
report = show_retrieval(engine, QUESTIONS, top=5)

In [ ]:
# T3. (optional) Full answers, exactly as the site runs them — ~15 s and ~2,000 Groq tokens per question
engine_llm = make_engine(with_llm=True)
answers = show_answers(engine_llm, QUESTIONS)

## The 50 test questions — chunks, then answer

Run steps 1 and 2, then **Q0** once. For each question:
- the **chunks** cell shows the 5 closest chunks, which ones are **→ sent** to the model, and the full passages (no Groq call);
- the **answer** cell shows the model's answer and each quote (✓ found in the book / ✗ not found).

Questions 48–50 are outside the books: the right result is «لم يرد في النص…».
"Runtime → Run all" works too: answer cells wait 15 s between Groq calls (about 15 min for all 50).

In [ ]:
# Q0. Setup for the 50 questions — run once (after steps 1 and 2). Read-only.
import os, sys, time
sys.path.insert(0, os.getcwd())   # the backend/ folder (step 1 moved there)
from scripts.inspect_questions import make_engine, show_retrieval, show_answers

Q50 = [
    "ما هو الوقت المختار لصلاة الظهر؟",   # 1
    "متى يبدأ وقت صلاة العصر ومتى ينتهي؟",   # 2
    "هل لصلاة المغرب وقت واحد أم أكثر؟",   # 3
    "ما هو الشفق الذي يدخل به وقت العشاء؟",   # 4
    "في أي الأوقات لا تجوز صلاة النافلة؟",   # 5
    "ما حكم الأذان في المساجد؟",   # 6
    "هل يجوز الأذان قبل دخول الوقت؟",   # 7
    "ما صفة الإقامة؟",   # 8
    "هل على المرأة أذان أو إقامة؟",   # 9
    "ما هي فرائض الوضوء؟",   # 10
    "ما هي نواقض الوضوء؟",   # 11
    "ماذا يفعل من نسي فرضا من فرائض الوضوء؟",   # 12
    "ما هي فرائض الغسل؟",   # 13
    "متى يجوز للحاضر الصحيح أن يتيمم؟",   # 14
    "ما هي فرائض التيمم؟",   # 15
    "كم أكثر مدة الحيض للمبتدأة؟",   # 16
    "كم أكثر مدة النفاس؟",   # 17
    "ما هي شروط الصلاة؟",   # 18
    "ما هي عورة المرأة في الصلاة؟",   # 19
    "ماذا يفعل من صلى إلى غير القبلة خطأ؟",   # 20
    "ما هي سنن الصلاة؟",   # 21
    "ما هي فضائل الصلاة؟",   # 22
    "ما الذي يكره فعله في الصلاة؟",   # 23
    "كيف يصلي المريض الذي لا يقدر على القيام؟",   # 24
    "ماذا يقول المصلي في الركوع والسجود؟",   # 25
    "متى يكون القنوت في صلاة الصبح؟",   # 26
    "كيف يجلس المصلي في التشهد؟",   # 27
    "كم تسليمة يسلم المأموم؟",   # 28
    "من أولى الناس بالإمامة؟",   # 29
    "أين يقف الرجل الواحد إذا صلى مع الإمام؟",   # 30
    "بماذا يدرك المسبوق فضل الجماعة؟",   # 31
    "متى يكون سجود السهو بعد السلام؟",   # 32
    "ما حكم من تكلم في الصلاة ساهيا؟",   # 33
    "ما حكم من نفخ في صلاته؟",   # 34
    "هل يسجد المسبوق مع الإمام لسهوه؟",   # 35
    "ماذا يفعل من نسي ركعة وتذكرها بعد السلام؟",   # 36
    "كيف تقضى الصلوات الفائتة؟",   # 37
    "هل يجوز لمن عليه صلوات فائتة أن يصلي النوافل؟",   # 38
    "متى يتم المسافر صلاته إذا نوى الإقامة في مكان؟",   # 39
    "على من تجب صلاة الجمعة؟",   # 40
    "ما حكم الغسل لصلاة الجمعة؟",   # 41
    "كيف تصلى صلاة الخوف في السفر؟",   # 42
    "ما حكم صلاة العيدين وهل لها أذان وإقامة؟",   # 43
    "كيف تصلى صلاة خسوف الشمس؟",   # 44
    "كم تكبيرة في صلاة الجنازة وأين يقف الإمام من الميت؟",   # 45
    "نسيت الفاتحة في ركعة من صلاة الظهر، شنوة نعمل؟",   # 46
    "الإمام قام لركعة خامسة، نقعد ولا نقوم معاه؟",   # 47
    "ما مقدار زكاة الفطر؟",   # 48
    "ما حكم صيام يوم عاشوراء؟",   # 49
    "ما هي شروط صحة البيع؟",   # 50
]
assert len(Q50) == 50

engine = make_engine(with_llm=False)     # chunks: no Groq calls
engine_llm = make_engine(with_llm=True)  # answers: ~2,000 Groq tokens each
_last_llm = [0.0]

def chunks(n):
    """Question n: the 5 closest chunks, which ones are sent to the model, and the full passages sent."""
    print(f"Q{n}. {Q50[n-1]}")
    show_retrieval(engine, [Q50[n-1]], top=5)

def answer(n):
    """Question n: the model's answer and its quotes, exactly as the site runs it.
    Waits so two answers are at least 15 s apart (Groq free limit: 8,000 tokens/minute)."""
    wait = 15 - (time.time() - _last_llm[0])
    if wait > 0:
        time.sleep(wait)
    print(f"Q{n}. {Q50[n-1]}")
    show_answers(engine_llm, [Q50[n-1]], sleep=0)
    _last_llm[0] = time.time()

print(len(Q50), "questions ready")

### أوقات الصلاة

In [ ]:
# Q1 · chunks — ما هو الوقت المختار لصلاة الظهر؟
chunks(1)

In [ ]:
# Q1 · answer — ما هو الوقت المختار لصلاة الظهر؟
answer(1)

In [ ]:
# Q2 · chunks — متى يبدأ وقت صلاة العصر ومتى ينتهي؟
chunks(2)

In [ ]:
# Q2 · answer — متى يبدأ وقت صلاة العصر ومتى ينتهي؟
answer(2)

In [ ]:
# Q3 · chunks — هل لصلاة المغرب وقت واحد أم أكثر؟
chunks(3)

In [ ]:
# Q3 · answer — هل لصلاة المغرب وقت واحد أم أكثر؟
answer(3)

In [ ]:
# Q4 · chunks — ما هو الشفق الذي يدخل به وقت العشاء؟
chunks(4)

In [ ]:
# Q4 · answer — ما هو الشفق الذي يدخل به وقت العشاء؟
answer(4)

In [ ]:
# Q5 · chunks — في أي الأوقات لا تجوز صلاة النافلة؟
chunks(5)

In [ ]:
# Q5 · answer — في أي الأوقات لا تجوز صلاة النافلة؟
answer(5)

### الأذان والإقامة

In [ ]:
# Q6 · chunks — ما حكم الأذان في المساجد؟
chunks(6)

In [ ]:
# Q6 · answer — ما حكم الأذان في المساجد؟
answer(6)

In [ ]:
# Q7 · chunks — هل يجوز الأذان قبل دخول الوقت؟
chunks(7)

In [ ]:
# Q7 · answer — هل يجوز الأذان قبل دخول الوقت؟
answer(7)

In [ ]:
# Q8 · chunks — ما صفة الإقامة؟
chunks(8)

In [ ]:
# Q8 · answer — ما صفة الإقامة؟
answer(8)

In [ ]:
# Q9 · chunks — هل على المرأة أذان أو إقامة؟
chunks(9)

In [ ]:
# Q9 · answer — هل على المرأة أذان أو إقامة؟
answer(9)

### الطهارة

In [ ]:
# Q10 · chunks — ما هي فرائض الوضوء؟
chunks(10)

In [ ]:
# Q10 · answer — ما هي فرائض الوضوء؟
answer(10)

In [ ]:
# Q11 · chunks — ما هي نواقض الوضوء؟
chunks(11)

In [ ]:
# Q11 · answer — ما هي نواقض الوضوء؟
answer(11)

In [ ]:
# Q12 · chunks — ماذا يفعل من نسي فرضا من فرائض الوضوء؟
chunks(12)

In [ ]:
# Q12 · answer — ماذا يفعل من نسي فرضا من فرائض الوضوء؟
answer(12)

In [ ]:
# Q13 · chunks — ما هي فرائض الغسل؟
chunks(13)

In [ ]:
# Q13 · answer — ما هي فرائض الغسل؟
answer(13)

In [ ]:
# Q14 · chunks — متى يجوز للحاضر الصحيح أن يتيمم؟
chunks(14)

In [ ]:
# Q14 · answer — متى يجوز للحاضر الصحيح أن يتيمم؟
answer(14)

In [ ]:
# Q15 · chunks — ما هي فرائض التيمم؟
chunks(15)

In [ ]:
# Q15 · answer — ما هي فرائض التيمم؟
answer(15)

In [ ]:
# Q16 · chunks — كم أكثر مدة الحيض للمبتدأة؟
chunks(16)

In [ ]:
# Q16 · answer — كم أكثر مدة الحيض للمبتدأة؟
answer(16)

In [ ]:
# Q17 · chunks — كم أكثر مدة النفاس؟
chunks(17)

In [ ]:
# Q17 · answer — كم أكثر مدة النفاس؟
answer(17)

### شروط الصلاة وسننها وفضائلها

In [ ]:
# Q18 · chunks — ما هي شروط الصلاة؟
chunks(18)

In [ ]:
# Q18 · answer — ما هي شروط الصلاة؟
answer(18)

In [ ]:
# Q19 · chunks — ما هي عورة المرأة في الصلاة؟
chunks(19)

In [ ]:
# Q19 · answer — ما هي عورة المرأة في الصلاة؟
answer(19)

In [ ]:
# Q20 · chunks — ماذا يفعل من صلى إلى غير القبلة خطأ؟
chunks(20)

In [ ]:
# Q20 · answer — ماذا يفعل من صلى إلى غير القبلة خطأ؟
answer(20)

In [ ]:
# Q21 · chunks — ما هي سنن الصلاة؟
chunks(21)

In [ ]:
# Q21 · answer — ما هي سنن الصلاة؟
answer(21)

In [ ]:
# Q22 · chunks — ما هي فضائل الصلاة؟
chunks(22)

In [ ]:
# Q22 · answer — ما هي فضائل الصلاة؟
answer(22)

In [ ]:
# Q23 · chunks — ما الذي يكره فعله في الصلاة؟
chunks(23)

In [ ]:
# Q23 · answer — ما الذي يكره فعله في الصلاة؟
answer(23)

In [ ]:
# Q24 · chunks — كيف يصلي المريض الذي لا يقدر على القيام؟
chunks(24)

In [ ]:
# Q24 · answer — كيف يصلي المريض الذي لا يقدر على القيام؟
answer(24)

### صفة الصلاة

In [ ]:
# Q25 · chunks — ماذا يقول المصلي في الركوع والسجود؟
chunks(25)

In [ ]:
# Q25 · answer — ماذا يقول المصلي في الركوع والسجود؟
answer(25)

In [ ]:
# Q26 · chunks — متى يكون القنوت في صلاة الصبح؟
chunks(26)

In [ ]:
# Q26 · answer — متى يكون القنوت في صلاة الصبح؟
answer(26)

In [ ]:
# Q27 · chunks — كيف يجلس المصلي في التشهد؟
chunks(27)

In [ ]:
# Q27 · answer — كيف يجلس المصلي في التشهد؟
answer(27)

In [ ]:
# Q28 · chunks — كم تسليمة يسلم المأموم؟
chunks(28)

In [ ]:
# Q28 · answer — كم تسليمة يسلم المأموم؟
answer(28)

### الإمامة والجماعة

In [ ]:
# Q29 · chunks — من أولى الناس بالإمامة؟
chunks(29)

In [ ]:
# Q29 · answer — من أولى الناس بالإمامة؟
answer(29)

In [ ]:
# Q30 · chunks — أين يقف الرجل الواحد إذا صلى مع الإمام؟
chunks(30)

In [ ]:
# Q30 · answer — أين يقف الرجل الواحد إذا صلى مع الإمام؟
answer(30)

In [ ]:
# Q31 · chunks — بماذا يدرك المسبوق فضل الجماعة؟
chunks(31)

In [ ]:
# Q31 · answer — بماذا يدرك المسبوق فضل الجماعة؟
answer(31)

### السهو

In [ ]:
# Q32 · chunks — متى يكون سجود السهو بعد السلام؟
chunks(32)

In [ ]:
# Q32 · answer — متى يكون سجود السهو بعد السلام؟
answer(32)

In [ ]:
# Q33 · chunks — ما حكم من تكلم في الصلاة ساهيا؟
chunks(33)

In [ ]:
# Q33 · answer — ما حكم من تكلم في الصلاة ساهيا؟
answer(33)

In [ ]:
# Q34 · chunks — ما حكم من نفخ في صلاته؟
chunks(34)

In [ ]:
# Q34 · answer — ما حكم من نفخ في صلاته؟
answer(34)

In [ ]:
# Q35 · chunks — هل يسجد المسبوق مع الإمام لسهوه؟
chunks(35)

In [ ]:
# Q35 · answer — هل يسجد المسبوق مع الإمام لسهوه؟
answer(35)

In [ ]:
# Q36 · chunks — ماذا يفعل من نسي ركعة وتذكرها بعد السلام؟
chunks(36)

In [ ]:
# Q36 · answer — ماذا يفعل من نسي ركعة وتذكرها بعد السلام؟
answer(36)

### قضاء الفوائت

In [ ]:
# Q37 · chunks — كيف تقضى الصلوات الفائتة؟
chunks(37)

In [ ]:
# Q37 · answer — كيف تقضى الصلوات الفائتة؟
answer(37)

In [ ]:
# Q38 · chunks — هل يجوز لمن عليه صلوات فائتة أن يصلي النوافل؟
chunks(38)

In [ ]:
# Q38 · answer — هل يجوز لمن عليه صلوات فائتة أن يصلي النوافل؟
answer(38)

### السفر والجمعة والخوف

In [ ]:
# Q39 · chunks — متى يتم المسافر صلاته إذا نوى الإقامة في مكان؟
chunks(39)

In [ ]:
# Q39 · answer — متى يتم المسافر صلاته إذا نوى الإقامة في مكان؟
answer(39)

In [ ]:
# Q40 · chunks — على من تجب صلاة الجمعة؟
chunks(40)

In [ ]:
# Q40 · answer — على من تجب صلاة الجمعة؟
answer(40)

In [ ]:
# Q41 · chunks — ما حكم الغسل لصلاة الجمعة؟
chunks(41)

In [ ]:
# Q41 · answer — ما حكم الغسل لصلاة الجمعة؟
answer(41)

In [ ]:
# Q42 · chunks — كيف تصلى صلاة الخوف في السفر؟
chunks(42)

In [ ]:
# Q42 · answer — كيف تصلى صلاة الخوف في السفر؟
answer(42)

### العيدان والخسوف والجنائز

In [ ]:
# Q43 · chunks — ما حكم صلاة العيدين وهل لها أذان وإقامة؟
chunks(43)

In [ ]:
# Q43 · answer — ما حكم صلاة العيدين وهل لها أذان وإقامة؟
answer(43)

In [ ]:
# Q44 · chunks — كيف تصلى صلاة خسوف الشمس؟
chunks(44)

In [ ]:
# Q44 · answer — كيف تصلى صلاة خسوف الشمس؟
answer(44)

In [ ]:
# Q45 · chunks — كم تكبيرة في صلاة الجنازة وأين يقف الإمام من الميت؟
chunks(45)

In [ ]:
# Q45 · answer — كم تكبيرة في صلاة الجنازة وأين يقف الإمام من الميت؟
answer(45)

### بالدارجة التونسية

In [ ]:
# Q46 · chunks — نسيت الفاتحة في ركعة من صلاة الظهر، شنوة نعمل؟
chunks(46)

In [ ]:
# Q46 · answer — نسيت الفاتحة في ركعة من صلاة الظهر، شنوة نعمل؟
answer(46)

In [ ]:
# Q47 · chunks — الإمام قام لركعة خامسة، نقعد ولا نقوم معاه؟
chunks(47)

In [ ]:
# Q47 · answer — الإمام قام لركعة خامسة، نقعد ولا نقوم معاه؟
answer(47)

### خارج النطاق — يجب أن يمتنع

In [ ]:
# Q48 · chunks — ما مقدار زكاة الفطر؟
chunks(48)

In [ ]:
# Q48 · answer — ما مقدار زكاة الفطر؟
answer(48)

In [ ]:
# Q49 · chunks — ما حكم صيام يوم عاشوراء؟
chunks(49)

In [ ]:
# Q49 · answer — ما حكم صيام يوم عاشوراء؟
answer(49)

In [ ]:
# Q50 · chunks — ما هي شروط صحة البيع؟
chunks(50)

In [ ]:
# Q50 · answer — ما هي شروط صحة البيع؟
answer(50)

## The learning loop

1. Users press 👎 under a wrong answer, say what's wrong and pick the correct sentence in the reader.
2. **Step 7** lists the new reports. Read each one.
3. **Step 8**: accept the good ones (that creates a *correction*: a worked example the model sees for similar questions) or reject them.
4. Copy the printed line into `backend/eval/golden.jsonl` and push: the mistake becomes a permanent test.
5. Re-run **step 5** to check the score went up and nothing else went down.

Corrections are only used by the site once `backend/app/learning.py` is written and `USE_CORRECTIONS=true` is set on Vercel.

In [ ]:
# 6. History of the evaluation runs (newest first, read-only)
import json, os, httpx
key = os.environ["SUPABASE_SERVICE_KEY"]
h = {"apikey": key, **({"Authorization": f"Bearer {key}"} if key.startswith("eyJ") else {})}
runs = httpx.get(os.environ["SUPABASE_URL"] + "/rest/v1/eval_runs", headers=h,
                 params={"select": "id,created_at,label,scores", "order": "created_at.desc", "limit": "10"}).json()
for r in runs:
    print(f"#{r['id']}  {r['created_at'][:16]}  {r['label'] or '':30s}  {json.dumps(r['scores'], ensure_ascii=False)}")

In [ ]:
# 7. New feedback reports from the site (read-only)
!python -m scripts.review_feedback list

In [ ]:
# 8. Accept or reject a report — edit the id and the short correct answer, then run.
#    accept: evidence = the sentence the user picked (or add --evidence "…" to set it yourself)
#    It prints the line to add to backend/eval/golden.jsonl.
!python -m scripts.review_feedback accept 1 --answer "الجواب القصير الصحيح كما في الكتاب."
# !python -m scripts.review_feedback reject 2

## After evaluating

If step 4 suggests a different `MIN_COSINE_SCORE` (or you want to change `COSINE_MARGIN`, `MAX_K`…):
Vercel → your project → **Settings → Environment Variables** → add/edit the variable → **Deployments → Redeploy**.

- *Hit* `NO` for an in-scope question → the right text is not reaching the model: raise `COSINE_MARGIN` a little (e.g. 0.08) or `MAX_K` to 4.
- Out-of-scope questions answered → raise `MIN_COSINE_SCORE`.
- In-scope questions refused («لم يرد في النص…» with `abstained=retrieval`) → lower `MIN_COSINE_SCORE`.